# IJIES — Post-Publication Technical Re-examination: Real-Time Video Evaluation

This notebook implements the post-publication technical re-examination of the real-time video experiment reported in **“Deep Learning Based Real-time Recognition of the ASL Alphabet from Hand Gestures.”**

The complete sequence is evaluated on one Tesla T4:

`YOLOv8 Stage-1 → Custom CNN → EfficientNetB0 → ResNet50V2 → ViT → YOLOv11m-cls → YOLOv11m-det → YOLOv12x → RT-DETR-L`

### Re-evaluation principles

- the same frame-level YOLOv8 Stage-1 localization output is reused by all downstream models;
- **strict Frame Accuracy (FA)** is computed over all frames, so missing Stage-1 localization is counted as a strict failure;
- the optional no-hand → `nothing` interpretation is reported separately as an operational metric;
- classification Temporal Stability (TS) is derived from the **shared Stage-1 YOLOv8 boxes**, not from classifier-specific boxes;
- target-detector TS uses target boxes correctly mapped from the resized 224×224 ROI back to full-frame coordinates;
- ViT video preprocessing uses the same `Normalize([0.5]*3, [0.5]*3)` protocol that reproduced the image-level benchmark/cross-domain checkpoint;
- Ultralytics label alias `del` is canonicalized to `delete`, while raw checkpoint class-name metadata are preserved in a separate audit;
- all checkpoint hashes, input-video hashes, subprocess logs, frame-level outputs, and summary files are retained;
- timing is reported transparently as **target-model-only FPS** and a **measured component-sum pipeline FPS estimate**. The notebook does not mislabel the latter as exact integrated wall-clock end-to-end FPS.

### EfficientNetB0 compatibility note

The archived EfficientNet checkpoint contains a serialized no-weight Python `Lambda` preprocessing layer that is incompatible with the current Keras runtime. This notebook performs a documented **post-publication runtime compatibility reconstruction** by replacing only that Lambda with an equivalent Keras-3-compatible layer implementing `preprocess_input(x * 255.0)`. Model weights are unchanged and independently hashed.


In [ ]:
# ============================================================
# 0. INSTALL + PARENT-ONLY IMPORTS
# ============================================================

%pip install -q ultralytics timm

import os
import sys
import json
import time
import shutil
import hashlib
import zipfile
import subprocess
import gc
import platform
import re
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd

print("Python:", sys.version)

# ------------------------------------------------------------
# Use ONE Tesla T4 only for all final timing/evaluation.
# ------------------------------------------------------------
r = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.used",
        "--format=csv,noheader"
    ],
    capture_output=True,
    text=True,
    check=False
)

print("\nGPU inventory:")
print(r.stdout.strip())

T4S = []

for line in r.stdout.splitlines():
    if not line.strip():
        continue

    parts = [
        x.strip()
        for x in line.split(",")
    ]

    if len(parts) < 2:
        continue

    gid = int(parts[0])
    name = parts[1]

    if "T4" in name.upper():
        T4S.append(
            (gid,name)
        )

if not T4S:
    raise RuntimeError(
        "❌ Cần Tesla T4 để tái đánh giá video theo môi trường đã công bố."
    )

TEST_GPU_ID, TEST_GPU_NAME = T4S[0]

print(
    f"\n✅ Final evaluation GPU: "
    f"{TEST_GPU_ID} - {TEST_GPU_NAME}"
)


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================

EVALUATION_VERSION = "IJIES_EDITOR_VIDEO_REEVAL_V3_2026_09"

WORK_ROOT = Path(
    "/kaggle/working/"
    "IJIES_EDITOR_VIDEO_REEVALUATION_V3"
)

WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_SUMMARY_CSV = WORK_ROOT / "IJIES_EDITOR_VIDEO_FINAL_SUMMARY.csv"
LABEL_AUDIT_CSV = WORK_ROOT / "IJIES_VIDEO_MODEL_LABEL_AUDIT.csv"
MODEL_MANIFEST_CSV = WORK_ROOT / "IJIES_VIDEO_MODEL_MANIFEST.csv"
VIDEO_MANIFEST_CSV = WORK_ROOT / "IJIES_VIDEO_INPUT_MANIFEST.csv"
FINAL_VERIFICATION_JSON = WORK_ROOT / "IJIES_EDITOR_VIDEO_FINAL_VERIFICATION.json"
FINAL_EVIDENCE_ZIP = Path("/kaggle/working/IJIES_EDITOR_VIDEO_EVIDENCE_PACKAGE.zip")

MODEL_ROOT = Path(
    "/kaggle/input/datasets/toilaxien/"
    "asl-models-realtime-test"
)

VIDEO_ROOT = Path(
    "/kaggle/input/datasets/toilaxien/"
    "asl-realtime-test-videos"
)

HAND_DETECTOR_PATH = Path(
    "/kaggle/input/models/toilaxien/"
    "yolov8-hand/gemmacpp/default/1/"
    "hand_yolov8s.pt"
)

HAND_CONF = 0.25
HAND_IMGSZ = 640
FRAME_STRIDE = 1

# Re-evaluation semantics locked for editor-facing evidence.
VIT_INPUT_SIZE = 224
VIT_MEAN = [0.5, 0.5, 0.5]
VIT_STD = [0.5, 0.5, 0.5]
LABEL_ALIASES = {"del": "delete"}
TARGET_INPUT_SIZE = 224


LABELS = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N",
    "O","P","Q","R","S","T","U","V","W","X","Y","Z",
    "delete","nothing","space"
]

CLASSIFICATION_MODELS = [
    ("Custom_CNN", MODEL_ROOT / "custom_cnn_asl.keras"),
    ("EfficientNetB0", MODEL_ROOT / "efficientnetb0_asl.keras"),
    ("ResNet50V2", MODEL_ROOT / "resnet50v2_asl.keras"),
    ("ViT", MODEL_ROOT / "ViT.pth"),
    ("YOLOv11m_cls", MODEL_ROOT / "YoLo11m_csf.pt"),
]

DETECTION_MODELS = [
    ("YOLOv11m_det", MODEL_ROOT / "YoLo11m_dectection.pt"),
    ("YOLOv12x", MODEL_ROOT / "YoLo12x.pt"),
    ("RTDETR_L", MODEL_ROOT / "rtdetr-l.pt"),
]

VIDEO_EXTS = {
    ".mov",".MOV",
    ".mp4",".MP4",
    ".avi",".AVI",
    ".mkv",".MKV"
}

VIDEOS = sorted([
    p
    for p in VIDEO_ROOT.rglob("*")
    if p.is_file()
    and p.suffix in VIDEO_EXTS
])

print("Videos:", len(VIDEOS))

if len(VIDEOS) != 29:
    print(
        "⚠ Published protocol describes 29 videos; "
        f"current dataset contains {len(VIDEOS)}."
    )

for name,path in (
    CLASSIFICATION_MODELS
    + DETECTION_MODELS
):
    if not path.exists():
        raise FileNotFoundError(
            f"{name}: {path}"
        )

if not HAND_DETECTOR_PATH.exists():
    raise FileNotFoundError(
        HAND_DETECTOR_PATH
    )


In [ ]:
# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(
    path,
    chunk_size=1024*1024
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:
        for chunk in iter(
            lambda:
                f.read(
                    chunk_size
                ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


def save_json(
    obj,
    path
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str
        ),
        encoding="utf-8"
    )


def run_sequential_child(
    name,
    cmd,
    extra_env=None,
    timeout=None
):
    env = os.environ.copy()

    env[
        "CUDA_VISIBLE_DEVICES"
    ] = str(
        TEST_GPU_ID
    )

    env[
        "TF_FORCE_GPU_ALLOW_GROWTH"
    ] = "true"

    env[
        "TF_CPP_MIN_LOG_LEVEL"
    ] = "2"

    env[
        "TF_ENABLE_ONEDNN_OPTS"
    ] = "0"

    env[
        "PYTORCH_CUDA_ALLOC_CONF"
    ] = "expandable_segments:True"

    if extra_env:
        env.update(
            {
                str(k):str(v)
                for k,v
                in extra_env.items()
            }
        )

    print(
        "\n"
        + "="*80
    )

    print(
        "START:",
        name
    )

    print(
        "="*80
    )

    t0 = time.perf_counter()

    try:
        p = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            env=env,
            timeout=timeout,
            check=False
        )

        rc = p.returncode

        stdout = p.stdout
        stderr = p.stderr

    except subprocess.TimeoutExpired as e:
        rc = "TIMEOUT"
        stdout = e.stdout or ""
        stderr = e.stderr or ""

    elapsed = (
        time.perf_counter()
        - t0
    )

    print(
        "returncode:",
        rc
    )

    print(
        "wall_seconds:",
        round(
            elapsed,
            2
        )
    )

    tail = (
        stderr
        if stderr
        else stdout
    )

    if tail:
        print(
            tail[-3000:]
        )

    # Persist subprocess evidence for editorial reproducibility.
    log_dir = WORK_ROOT / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))
    (log_dir / f"{safe_name}.command.txt").write_text(
        " ".join(map(str, cmd)), encoding="utf-8"
    )
    (log_dir / f"{safe_name}.stdout.txt").write_text(
        stdout or "", encoding="utf-8"
    )
    (log_dir / f"{safe_name}.stderr.txt").write_text(
        stderr or "", encoding="utf-8"
    )

    # Child has exited; its CUDA context is now released.
    gc.collect()

    time.sleep(
        2
    )

    return {
        "name":
            name,

        "returncode":
            rc,

        "wall_seconds":
            elapsed,

        "stdout":
            stdout,

        "stderr":
            stderr,
    }


## Reproducibility manifest and semantic-label audit

This section records file hashes, package versions, video-input hashes, and the class-name metadata stored in each Ultralytics checkpoint.

The audit applies only semantic canonicalization that is explicitly documented in this notebook:

- `del` → `delete`
- one-character alphabet labels → uppercase
- `delete`, `nothing`, and `space` → lowercase

No model predictions are changed beyond this string-level label canonicalization. The raw checkpoint class names and the canonicalized names are both preserved in the audit outputs.


In [ ]:
# ============================================================
# 2B. REPRODUCIBILITY MANIFEST + ULTRALYTICS LABEL AUDIT
# ============================================================

def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

environment_manifest = {
    "evaluation_version": EVALUATION_VERSION,
    "python": sys.version,
    "platform": platform.platform(),
    "gpu_selected": TEST_GPU_NAME,
    "tensorflow": pkg_version("tensorflow"),
    "keras": pkg_version("keras"),
    "torch": pkg_version("torch"),
    "torchvision": pkg_version("torchvision"),
    "timm": pkg_version("timm"),
    "ultralytics": pkg_version("ultralytics"),
    "numpy": pkg_version("numpy"),
    "pandas": pkg_version("pandas"),
    "opencv_python": pkg_version("opencv-python"),
}
save_json(environment_manifest, WORK_ROOT / "environment_manifest.json")

model_rows = []
all_model_paths = [
    ("YOLOv8_Stage1", "shared_localization", HAND_DETECTOR_PATH),
    *[(n, "classification", p) for n, p in CLASSIFICATION_MODELS],
    *[(n, "detection", p) for n, p in DETECTION_MODELS],
]
for model_name, branch, model_path in all_model_paths:
    model_rows.append({
        "model": model_name,
        "branch": branch,
        "checkpoint_path": str(model_path),
        "checkpoint_size_bytes": int(model_path.stat().st_size),
        "checkpoint_sha256": sha256_file(model_path),
    })
model_manifest_df = pd.DataFrame(model_rows)
model_manifest_df.to_csv(MODEL_MANIFEST_CSV, index=False)
display(model_manifest_df)

video_rows = []
for vp in VIDEOS:
    video_rows.append({
        "video": vp.name,
        "path": str(vp),
        "size_bytes": int(vp.stat().st_size),
        "sha256": sha256_file(vp),
    })
video_manifest_df = pd.DataFrame(video_rows)
video_manifest_df.to_csv(VIDEO_MANIFEST_CSV, index=False)
print("Video manifest rows:", len(video_manifest_df))

LABEL_AUDIT_WORKER = Path("/kaggle/working/ijies_label_audit_worker.py")

label_audit_worker_source = r'''
import sys, json
from pathlib import Path

MODEL_NAME = sys.argv[1]
MODEL_PATH = Path(sys.argv[2])
OUT_JSON = Path(sys.argv[3])
EXPECTED = json.loads(sys.argv[4])

from ultralytics import YOLO
try:
    from ultralytics import RTDETR
except Exception:
    RTDETR = None

def normalize_label(x):
    s = str(x).strip()
    low = s.lower()
    if low == "del":
        return "delete"
    if low in {"delete", "nothing", "space"}:
        return low
    if len(s) == 1:
        return s.upper()
    return s

if MODEL_NAME == "RTDETR_L" and RTDETR is not None:
    try:
        model = RTDETR(str(MODEL_PATH))
    except Exception:
        model = YOLO(str(MODEL_PATH))
else:
    model = YOLO(str(MODEL_PATH))

names_obj = model.names
if isinstance(names_obj, dict):
    raw_names = [str(names_obj[k]) for k in sorted(names_obj)]
else:
    raw_names = [str(x) for x in names_obj]

canonical_names = [normalize_label(x) for x in raw_names]
expected_set = set(EXPECTED)
canonical_set = set(canonical_names)

payload = {
    "model": MODEL_NAME,
    "checkpoint": str(MODEL_PATH),
    "n_classes": len(raw_names),
    "raw_names": raw_names,
    "canonical_names": canonical_names,
    "has_del_raw": any(str(x).strip().lower() == "del" for x in raw_names),
    "has_delete_raw": any(str(x).strip().lower() == "delete" for x in raw_names),
    "missing_expected_labels_after_canonicalization": sorted(expected_set - canonical_set),
    "unexpected_labels_after_canonicalization": sorted(canonical_set - expected_set),
    "semantic_label_set_match": (
        len(raw_names) == len(EXPECTED)
        and canonical_set == expected_set
    ),
    "canonicalization_policy": (
        "del->delete; single letters upper-cased; "
        "delete/nothing/space lower-cased"
    ),
}

OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps(payload, indent=2))
'''

LABEL_AUDIT_WORKER.write_text(label_audit_worker_source, encoding="utf-8")

audit_dir = WORK_ROOT / "label_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

ultralytics_audit_models = [
    ("YOLOv11m_cls", dict(CLASSIFICATION_MODELS)["YOLOv11m_cls"]),
    *DETECTION_MODELS,
]

audit_status = []

for model_name, model_path in ultralytics_audit_models:
    out_json = audit_dir / f"{model_name}.json"
    cmd = [
        sys.executable,
        str(LABEL_AUDIT_WORKER),
        model_name,
        str(model_path),
        str(out_json),
        json.dumps(LABELS),
    ]
    result = run_sequential_child(
        f"label_audit::{model_name}",
        cmd,
        timeout=30*60
    )
    audit_status.append({
        "model": model_name,
        "returncode": result["returncode"],
        "json_exists": out_json.exists(),
    })

audit_rows = []
for model_name, _ in ultralytics_audit_models:
    p = audit_dir / f"{model_name}.json"
    if not p.exists():
        audit_rows.append({
            "model": model_name,
            "status": "FAILED",
            "semantic_label_set_match": False,
        })
        continue

    d = json.loads(p.read_text(encoding="utf-8"))
    audit_rows.append({
        "model": d["model"],
        "status": "PASS",
        "n_classes": d["n_classes"],
        "has_del_raw": d["has_del_raw"],
        "has_delete_raw": d["has_delete_raw"],
        "semantic_label_set_match": d["semantic_label_set_match"],
        "missing_expected_labels_after_canonicalization": ";".join(
            d["missing_expected_labels_after_canonicalization"]
        ),
        "unexpected_labels_after_canonicalization": ";".join(
            d["unexpected_labels_after_canonicalization"]
        ),
        "raw_names_json": json.dumps(d["raw_names"], ensure_ascii=False),
        "canonical_names_json": json.dumps(d["canonical_names"], ensure_ascii=False),
    })

label_audit_df = pd.DataFrame(audit_rows)
label_audit_df.to_csv(LABEL_AUDIT_CSV, index=False)
display(label_audit_df)

print("\nLabel-audit file:", LABEL_AUDIT_CSV)
print(
    "All Ultralytics semantic label sets match after canonicalization:",
    bool(label_audit_df["semantic_label_set_match"].fillna(False).all())
)


## EfficientNetB0 compatibility reconstruction

The original serialized `effnet_preprocess` Lambda is audited before replacement. Only this no-weight preprocessing layer is replaced. The transformation remains:

`input in [0,1] → x × 255 → tf.keras.applications.efficientnet.preprocess_input(x)`

The notebook records the original checkpoint hash, compatibility-archive hash, internal weights-file hash, and the decoded Lambda metadata when available.


In [ ]:
# ============================================================
# 3. BUILD EFFICIENTNET KAGGLE-COMPAT ARCHIVE
# Working fix validated on a single realtime clip.
# ============================================================

import marshal
import base64

EFF_ORIGINAL = dict(
    CLASSIFICATION_MODELS
)["EfficientNetB0"]

EFF_COMPAT = Path(
    "/kaggle/working/"
    "efficientnetb0_asl_kaggle_compat.keras"
)

EFF_TEMP = Path(
    "/kaggle/working/"
    "effnet_full_video_compat_build"
)

if EFF_TEMP.exists():
    shutil.rmtree(
        EFF_TEMP
    )

if EFF_COMPAT.exists():
    EFF_COMPAT.unlink()

EFF_TEMP.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Extract original archive
# ------------------------------------------------------------
with zipfile.ZipFile(
    EFF_ORIGINAL,
    "r"
) as z:
    z.extractall(
        EFF_TEMP
    )

config_path = (
    EFF_TEMP
    / "config.json"
)

config = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

target = None

for layer in config[
    "config"
][
    "layers"
]:
    if (
        layer.get(
            "class_name"
        ) == "Lambda"
        and
        layer.get(
            "config",
            {}
        ).get(
            "name"
        ) == "effnet_preprocess"
    ):
        target = layer
        break

if target is None:
    raise RuntimeError(
        "effnet_preprocess Lambda not found."
    )

print(
    "✅ Found Lambda:",
    target[
        "config"
    ][
        "name"
    ]
)

# ------------------------------------------------------------
# Audit serialized Lambda metadata
# ------------------------------------------------------------
fn = target[
    "config"
].get(
    "function",
    {}
)

fn_cfg = (
    fn.get(
        "config",
        {}
    )
    if isinstance(
        fn,
        dict
    )
    else {}
)

code_b64 = fn_cfg.get(
    "code"
)

lambda_audit = {
    "layer_name":
        target[
            "config"
        ].get(
            "name"
        ),

    "original_class_name":
        target.get(
            "class_name"
        ),

    "serialized_code_present":
        bool(
            code_b64
        ),

    "compatibility_replacement":
        (
            "EffNetPreprocessCompat implementing "
            "preprocess_input(x * 255.0)"
        ),
}

if code_b64:
    try:
        code_obj = marshal.loads(
            base64.b64decode(
                code_b64
            )
        )

        lambda_audit[
            "co_names"
        ] = list(
            code_obj.co_names
        )

        lambda_audit[
            "co_consts"
        ] = [
            repr(x)
            for x
            in code_obj.co_consts
        ]

        print(
            "Lambda co_names :",
            code_obj.co_names
        )

        print(
            "Lambda co_consts:",
            code_obj.co_consts
        )

        if (
            "preprocess_input"
            not in code_obj.co_names
        ):
            raise RuntimeError(
                "Serialized Lambda does not reference preprocess_input. "
                "Stop instead of guessing its semantics."
            )

    except RuntimeError:
        raise

    except Exception as e:
        print(
            "⚠ Could not decode Lambda bytecode:",
            repr(e)
        )

        lambda_audit[
            "bytecode_decode_error"
        ] = repr(
            e
        )

# ------------------------------------------------------------
# SHA256 of untouched weights inside extracted archive
# ------------------------------------------------------------
weights_candidates = sorted(
    EFF_TEMP.glob(
        "*.weights.h5"
    )
)

weights_sha256 = None
weights_filename = None

if weights_candidates:
    weights_file = weights_candidates[
        0
    ]

    weights_filename = (
        weights_file.name
    )

    weights_sha256 = sha256_file(
        weights_file
    )

    print(
        "Weights file:",
        weights_filename
    )

    print(
        "Weights SHA256:",
        weights_sha256
    )

# ------------------------------------------------------------
# Replace ONLY the no-weight serialized Lambda
# ------------------------------------------------------------
old_cfg = target[
    "config"
]

target[
    "module"
] = None

target[
    "class_name"
] = "EffNetPreprocessCompat"

target[
    "registered_name"
] = "EffNetPreprocessCompat"

target[
    "config"
] = {
    "name":
        old_cfg.get(
            "name",
            "effnet_preprocess"
        ),

    "trainable":
        old_cfg.get(
            "trainable",
            True
        ),

    "dtype":
        old_cfg.get(
            "dtype",
            "float32"
        ),
}

config_path.write_text(
    json.dumps(
        config
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# Repack archive
# ------------------------------------------------------------
with zipfile.ZipFile(
    EFF_COMPAT,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for p in EFF_TEMP.rglob(
        "*"
    ):
        if p.is_file():
            z.write(
                p,
                p.relative_to(
                    EFF_TEMP
                )
            )

lambda_audit.update({
    "original_model_path":
        str(
            EFF_ORIGINAL
        ),

    "original_model_sha256":
        sha256_file(
            EFF_ORIGINAL
        ),

    "compat_model_path":
        str(
            EFF_COMPAT
        ),

    "compat_model_sha256":
        sha256_file(
            EFF_COMPAT
        ),

    "weights_filename":
        weights_filename,

    "weights_sha256":
        weights_sha256,

    "weights_modified":
        False,

    "scientific_status":
        "POST_PUBLICATION_RUNTIME_COMPATIBILITY_RECONSTRUCTION",
})

save_json(
    lambda_audit,
    WORK_ROOT
    / "efficientnet_compatibility_audit.json"
)

print(
    "\n✅ EfficientNet Kaggle compatibility archive:"
)

print(
    EFF_COMPAT
)

print(
    "Size MB:",
    EFF_COMPAT.stat().st_size
    / 1024
    / 1024
)


In [ ]:
# ============================================================
# 4. WRITE ISOLATED WORKERS
# ============================================================

STAGE1_WORKER = Path(
    "/kaggle/working/"
    "ijies_seq_stage1_worker.py"
)

CLS_WORKER = Path(
    "/kaggle/working/"
    "ijies_seq_classifier_worker.py"
)

DET_WORKER = Path(
    "/kaggle/working/"
    "ijies_seq_detector_worker.py"
)

STAGE1_WORKER.write_text(
    '\nimport sys, json, time, importlib.metadata\nfrom pathlib import Path\n\nMODEL_PATH = Path(sys.argv[1])\nPARTS_DIR = Path(sys.argv[2])\nCONF = float(sys.argv[3])\nIMGSZ = int(sys.argv[4])\nSTRIDE = int(sys.argv[5])\nVIDEOS = [Path(x) for x in json.loads(sys.argv[6])]\n\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport torch\nfrom ultralytics import YOLO\n\nPARTS_DIR.mkdir(parents=True, exist_ok=True)\nFAIL_DIR = PARTS_DIR.parent / "failure_samples"\nFAIL_DIR.mkdir(parents=True, exist_ok=True)\n\nGPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\nULTRALYTICS_VERSION = importlib.metadata.version("ultralytics")\n\ndef iou(a, b):\n    if a is None or b is None:\n        return None\n    xA=max(a[0],b[0]); yA=max(a[1],b[1])\n    xB=min(a[2],b[2]); yB=min(a[3],b[3])\n    inter=max(0.0,xB-xA)*max(0.0,yB-yA)\n    aa=max(0.0,a[2]-a[0])*max(0.0,a[3]-a[1])\n    ab=max(0.0,b[2]-b[0])*max(0.0,b[3]-b[1])\n    den=aa+ab-inter\n    return inter/den if den>0 else 0.0\n\nmodel = YOLO(str(MODEL_PATH))\n\nfor vp in VIDEOS:\n    final_csv = PARTS_DIR / f"{vp.stem}.csv"\n\n    if final_csv.exists():\n        print("SKIP", vp.name)\n        continue\n\n    gt = vp.stem.upper() if len(vp.stem)==1 else vp.stem.lower()\n\n    cap = cv2.VideoCapture(str(vp))\n    idx = -1\n    prev_box = None\n    rows = []\n    saved_no_detection = 0\n    saved_multiple = 0\n\n    while True:\n        ok, frame = cap.read()\n        if not ok:\n            break\n\n        idx += 1\n        if idx % STRIDE != 0:\n            continue\n\n        h, w = frame.shape[:2]\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n        t0 = time.perf_counter()\n\n        result = model.predict(\n            frame,\n            imgsz=IMGSZ,\n            conf=CONF,\n            verbose=False,\n            device=0\n        )[0]\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n        t1 = time.perf_counter()\n\n        n_boxes = 0 if result.boxes is None else len(result.boxes)\n        detected = n_boxes > 0\n\n        box = None\n        det_conf = None\n        area_ratio = None\n\n        if detected:\n            boxes = result.boxes.xyxy.detach().cpu().numpy()\n            confs = result.boxes.conf.detach().cpu().numpy()\n\n            areas = (\n                (boxes[:,2]-boxes[:,0])\n                * (boxes[:,3]-boxes[:,1])\n            )\n\n            j = int(np.argmax(areas))\n            box = boxes[j].astype(float).tolist()\n            det_conf = float(confs[j])\n            area_ratio = float(\n                max(0.0, box[2]-box[0])\n                * max(0.0, box[3]-box[1])\n                / max(1.0, float(w*h))\n            )\n\n        pair = (\n            iou(prev_box, box)\n            if prev_box is not None and box is not None\n            else None\n        )\n\n        rows.append({\n            "video": vp.name,\n            "true_label": gt,\n            "frame_idx": idx,\n            "frame_width": w,\n            "frame_height": h,\n            "detected": bool(detected),\n            "n_detections": int(n_boxes),\n            "multiple_detections": bool(n_boxes > 1),\n            "det_conf": det_conf,\n            "bbox_area_ratio": area_ratio,\n            "x1": box[0] if box else np.nan,\n            "y1": box[1] if box else np.nan,\n            "x2": box[2] if box else np.nan,\n            "y2": box[3] if box else np.nan,\n            "stage1_latency_ms": (t1-t0)*1000.0,\n            "iou_with_prev_detected_pair": pair,\n            "iou_with_prev_zero_missing": (\n                (pair if pair is not None else 0.0)\n                if idx > 0 else np.nan\n            ),\n            "worker_gpu_name": GPU_NAME,\n            "ultralytics_version": ULTRALYTICS_VERSION,\n        })\n\n        if not detected and saved_no_detection < 10:\n            cv2.imwrite(\n                str(FAIL_DIR / f"NO_DET_{vp.stem}_{idx:06d}.jpg"),\n                frame\n            )\n            saved_no_detection += 1\n\n        if n_boxes > 1 and saved_multiple < 10:\n            cv2.imwrite(\n                str(FAIL_DIR / f"MULTI_DET_{vp.stem}_{idx:06d}.jpg"),\n                frame\n            )\n            saved_multiple += 1\n\n        prev_box = box\n\n    cap.release()\n\n    tmp = final_csv.with_suffix(".tmp.csv")\n    pd.DataFrame(rows).to_csv(tmp, index=False)\n    tmp.replace(final_csv)\n\n    print("DONE", vp.name, len(rows), GPU_NAME)\n\nprint("WORKER COMPLETE")\n',
    encoding="utf-8"
)

CLS_WORKER.write_text(
    '\nimport sys,json,time\nfrom pathlib import Path\n\nMODEL_NAME=sys.argv[1]\nMODEL_PATH=Path(sys.argv[2])\nSTAGE1_CSV=Path(sys.argv[3])\nVIDEOS=[Path(x) for x in json.loads(sys.argv[4])]\nOUT_ROOT=Path(sys.argv[5])\nSHARED_TS_JSON=Path(sys.argv[6])\n\nLABELS=[\n"A","B","C","D","E","F","G","H","I","J","K","L","M","N",\n"O","P","Q","R","S","T","U","V","W","X","Y","Z",\n"delete","nothing","space"\n]\n\nimport numpy as np\nimport pandas as pd\nimport cv2\n\nshared_ts=json.loads(SHARED_TS_JSON.read_text())\nstage1=pd.read_csv(STAGE1_CSV)\n\nlookup={\n(r.video,int(r.frame_idx)):r\nfor r in stage1.itertuples(index=False)\n}\n\nout_dir=OUT_ROOT/"classifiers"/MODEL_NAME\nparts_dir=out_dir/"per_video"\nparts_dir.mkdir(parents=True,exist_ok=True)\n\nframework=None\nmodel=None\n\n\n\nif MODEL_NAME in {"Custom_CNN","EfficientNetB0","ResNet50V2"}:\n    import tensorflow as tf\n    from tensorflow import keras\n\n    gpus=tf.config.list_physical_devices(\n        "GPU"\n    )\n\n    if gpus:\n        try:\n            tf.config.experimental.set_memory_growth(\n                gpus[0],\n                True\n            )\n        except RuntimeError:\n            pass\n\n    class Cast(\n        tf.keras.layers.Layer\n    ):\n        def __init__(\n            self,\n            dtype="float32",\n            **kwargs\n        ):\n            super().__init__(\n                **kwargs\n            )\n            self._dtype=dtype\n\n        def call(\n            self,\n            inputs\n        ):\n            return tf.cast(\n                inputs,\n                self._dtype\n            )\n\n        def get_config(\n            self\n        ):\n            cfg=super().get_config()\n\n            cfg.update({\n                "dtype":\n                    self._dtype\n            })\n\n            return cfg\n\n    class EffNetPreprocessCompat(\n        tf.keras.layers.Layer\n    ):\n        """\n        Safe Keras-3 replacement for the old serialized Lambda:\n            lambda z: preprocess_input(z * 255.0)\n\n        No trainable weights.\n        """\n        def __init__(\n            self,\n            **kwargs\n        ):\n            super().__init__(\n                **kwargs\n            )\n\n            self.supports_masking=True\n\n        def call(\n            self,\n            inputs,\n            training=None,\n            mask=None,\n            **kwargs\n        ):\n            x=(\n                inputs\n                *\n                tf.cast(\n                    255.0,\n                    inputs.dtype\n                )\n            )\n\n            return (\n                tf.keras\n                .applications\n                .efficientnet\n                .preprocess_input(\n                    x\n                )\n            )\n\n        def compute_mask(\n            self,\n            inputs,\n            mask=None\n        ):\n            return mask\n\n        def get_config(\n            self\n        ):\n            return super().get_config()\n\n    custom_objects={\n        "Cast":\n            Cast\n    }\n\n    if MODEL_NAME=="EfficientNetB0":\n        custom_objects[\n            "EffNetPreprocessCompat"\n        ]=EffNetPreprocessCompat\n\n    print(\n        f"[{MODEL_NAME}] Loading model..."\n    )\n\n    model=keras.models.load_model(\n        str(\n            MODEL_PATH\n        ),\n        custom_objects=custom_objects,\n        compile=False,\n        safe_mode=False\n    )\n\n    print(\n        f"✅ {MODEL_NAME} deserialization PASS"\n    )\n\n    print(\n        "Input :",\n        model.input_shape\n    )\n\n    print(\n        "Output:",\n        model.output_shape\n    )\n\n    if MODEL_NAME=="EfficientNetB0":\n        print(\n            "[EfficientNetB0] Starting dummy forward smoke test..."\n        )\n\n        dummy=tf.zeros(\n            (\n                1,\n                224,\n                224,\n                3\n            ),\n            dtype=tf.float32\n        )\n\n        smoke_np=model(\n            dummy,\n            training=False\n        ).numpy()\n\n        if smoke_np.shape!=(\n            1,\n            29\n        ):\n            raise RuntimeError(\n                f"Unexpected EfficientNet output shape: {smoke_np.shape}"\n            )\n\n        if not np.isfinite(\n            smoke_np\n        ).all():\n            raise RuntimeError(\n                "EfficientNet smoke output contains NaN/Inf."\n            )\n\n        print(\n            "✅ EfficientNet forward-pass smoke test PASS"\n        )\n\n    framework="keras"\n\nelif MODEL_NAME=="ViT":\n    import torch,timm\n    from PIL import Image\n    from torchvision import transforms\n    device="cuda:0"\n\n    vit_transform=transforms.Compose([\n        transforms.Resize((224,224)),\n        transforms.ToTensor(),\n        transforms.Normalize(\n            mean=[0.5,0.5,0.5],\n            std=[0.5,0.5,0.5]\n        ),\n    ])\n\n    model=timm.create_model(\n        "vit_base_patch16_224",\n        pretrained=False,\n        num_classes=29\n    )\n\n    state=torch.load(str(MODEL_PATH),map_location="cpu")\n    if isinstance(state,dict) and "state_dict" in state:\n        state=state["state_dict"]\n    state={k.replace("module.",""):v for k,v in state.items()}\n\n    model.load_state_dict(state,strict=True)\n    model.eval().to(device)\n    framework="vit"\n\nelse:\n    import torch\n    from ultralytics import YOLO\n    model=YOLO(str(MODEL_PATH))\n    framework="yolo"\n\ndef normalize_label(x):\n    s=str(x).strip()\n    low=s.lower()\n    if low=="del":\n        return "delete"\n    if low in {"delete","nothing","space"}:\n        return low\n    if len(s)==1 and s.upper() in LABELS:\n        return s.upper()\n    return s\n\ndef predict(roi):\n    t_pre0=time.perf_counter()\n\n    if framework=="keras":\n        rgb=cv2.cvtColor(roi,cv2.COLOR_BGR2RGB)\n        x=cv2.resize(rgb,(224,224)).astype(np.float32)/255.0\n        x=np.expand_dims(x,0)\n        t_pre1=time.perf_counter()\n\n        t_model0=time.perf_counter()\n\n        if MODEL_NAME=="EfficientNetB0":\n            input_name=model.inputs[0].name.split(":")[0]\n            y=model(\n                {input_name:tf.convert_to_tensor(x)},\n                training=False\n            ).numpy()[0]\n        else:\n            y=model(\n                tf.convert_to_tensor(x),\n                training=False\n            ).numpy()[0]\n\n        t_model1=time.perf_counter()\n\n        t_post0=time.perf_counter()\n        i=int(np.argmax(y))\n        raw_pred=LABELS[i]\n        pred=normalize_label(raw_pred)\n        conf=float(y[i])\n        t_post1=time.perf_counter()\n\n    elif framework=="vit":\n        import torch\n\n        rgb=cv2.cvtColor(roi,cv2.COLOR_BGR2RGB)\n        x=vit_transform(Image.fromarray(rgb)).unsqueeze(0).to(device)\n        t_pre1=time.perf_counter()\n\n        torch.cuda.synchronize()\n        t_model0=time.perf_counter()\n\n        with torch.inference_mode():\n            y=torch.softmax(model(x)[0],dim=0)\n\n        torch.cuda.synchronize()\n        t_model1=time.perf_counter()\n\n        t_post0=time.perf_counter()\n        i=int(torch.argmax(y).item())\n        raw_pred=LABELS[i]\n        pred=normalize_label(raw_pred)\n        conf=float(y[i].item())\n        t_post1=time.perf_counter()\n\n    else:\n        import torch\n        resized=cv2.resize(roi,(224,224))\n        t_pre1=time.perf_counter()\n\n        torch.cuda.synchronize()\n        t_model0=time.perf_counter()\n\n        r=model.predict(\n            resized,\n            imgsz=224,\n            verbose=False,\n            device=0\n        )[0]\n\n        torch.cuda.synchronize()\n        t_model1=time.perf_counter()\n\n        t_post0=time.perf_counter()\n        i=int(torch.argmax(r.probs.data).item())\n        name=model.names[i] if isinstance(model.names,dict) else model.names[i]\n        raw_pred=str(name)\n        pred=normalize_label(raw_pred)\n        conf=float(r.probs.data[i].item())\n        t_post1=time.perf_counter()\n\n    return (\n        raw_pred,\n        pred,\n        conf,\n        (t_pre1-t_pre0)*1000.0,\n        (t_model1-t_model0)*1000.0,\n        (t_post1-t_post0)*1000.0,\n    )\n\nfor vp in VIDEOS:\n    final_csv=parts_dir/f"{vp.stem}.csv"\n\n    if final_csv.exists():\n        # Reuse only outputs created with the NOTHING-aware schema.\n        try:\n            cached_cols = set(pd.read_csv(final_csv, nrows=1).columns)\n        except Exception:\n            cached_cols = set()\n\n        required_cols = {\n            "pred_label_operational",\n            "correct_operational",\n            "operational_nothing_policy_applied",\n        }\n\n        if required_cols.issubset(cached_cols):\n            print("SKIP",vp.name)\n            continue\n\n        print("REBUILD old-schema cache:",vp.name)\n        final_csv.unlink()\n\n    gt=normalize_label(vp.stem)\n    cap=cv2.VideoCapture(str(vp))\n\n    idx=-1\n    rows=[]\n\n    while True:\n        ok,frame=cap.read()\n        if not ok:\n            break\n\n        idx+=1\n        s=lookup.get((vp.name,idx))\n        if s is None:\n            continue\n\n        if not bool(s.detected):\n            # ------------------------------------------------\n            # STRICT:\n            # Missing Stage-1 localization is always a failure.\n            # ------------------------------------------------\n            strict_pred="__NO_HAND__"\n            strict_correct=False\n\n            # ------------------------------------------------\n            # OPERATIONAL NOTHING POLICY:\n            # Only when GT is actually "nothing", a no-hand\n            # Stage-1 result is interpreted as system output\n            # "nothing". This metric is reported separately.\n            # ------------------------------------------------\n            operational_policy_applied=(gt=="nothing")\n\n            operational_pred=(\n                "nothing"\n                if operational_policy_applied\n                else "__NO_HAND__"\n            )\n\n            operational_correct=(\n                operational_pred==gt\n            )\n\n            rows.append({\n                "video":vp.name,\n                "frame_idx":idx,\n                "true_label":gt,\n                "stage1_detected":False,\n\n                # Strict output\n                "pred_label_raw":strict_pred,\n                "pred_label":strict_pred,\n                "label_alias_applied":False,\n                "pred_conf":np.nan,\n                "correct_strict":strict_correct,\n\n                # Operational system output\n                "pred_label_operational":operational_pred,\n                "correct_operational":operational_correct,\n                "operational_nothing_policy_applied":operational_policy_applied,\n\n                "localization_failure":True,\n\n                # Target classifier was not executed, therefore this is\n                # not counted as a recognition-model failure.\n                "recognition_failure":False,\n\n                "stage1_latency_ms":float(s.stage1_latency_ms),\n                "roi_crop_ms":0.0,\n                "preprocess_ms":0.0,\n                "model_call_ms":0.0,\n                "postprocess_ms":0.0,\n                "component_sum_e2e_ms":float(s.stage1_latency_ms),\n            })\n            continue\n\n        x1,y1,x2,y2=[int(round(v)) for v in [s.x1,s.y1,s.x2,s.y2]]\n        h,w=frame.shape[:2]\n\n        x1=max(0,min(x1,w-1)); x2=max(1,min(x2,w))\n        y1=max(0,min(y1,h-1)); y2=max(1,min(y2,h))\n\n        c0=time.perf_counter()\n        roi=frame[y1:y2,x1:x2]\n        c1=time.perf_counter()\n        crop_ms=(c1-c0)*1000.0\n\n        if roi.size==0:\n            raw_pred="__EMPTY_ROI__"\n            pred="__EMPTY_ROI__"\n            conf=np.nan\n            pre_ms=model_ms=post_ms=0.0\n        else:\n            raw_pred,pred,conf,pre_ms,model_ms,post_ms=predict(roi)\n\n        correct=pred==gt\n\n        rows.append({\n            "video":vp.name,\n            "frame_idx":idx,\n            "true_label":gt,\n            "stage1_detected":True,\n\n            # With a valid ROI, strict and operational outputs are identical.\n            "pred_label_raw":raw_pred,\n            "pred_label":pred,\n            "label_alias_applied":bool(str(raw_pred).strip().lower()=="del" and pred=="delete"),\n            "pred_conf":conf,\n            "correct_strict":correct,\n            "pred_label_operational":pred,\n            "correct_operational":correct,\n            "operational_nothing_policy_applied":False,\n\n            "localization_failure":False,\n            "recognition_failure":not correct,\n            "stage1_latency_ms":float(s.stage1_latency_ms),\n            "roi_crop_ms":crop_ms,\n            "preprocess_ms":pre_ms,\n            "model_call_ms":model_ms,\n            "postprocess_ms":post_ms,\n            "component_sum_e2e_ms":(\n                float(s.stage1_latency_ms)\n                + crop_ms + pre_ms + model_ms + post_ms\n            ),\n        })\n\n    cap.release()\n\n    tmp=final_csv.with_suffix(".tmp.csv")\n    pd.DataFrame(rows).to_csv(tmp,index=False)\n    tmp.replace(final_csv)\n\n    print("DONE",MODEL_NAME,vp.name)\n\ndf=pd.concat(\n    [pd.read_csv(parts_dir/f"{vp.stem}.csv") for vp in VIDEOS],\n    ignore_index=True\n)\n\ndf.to_csv(out_dir/"frame_level_outputs.csv",index=False)\n\nlocalized=df[df["stage1_detected"]==True]\n\nper_video=(\n    df.groupby("video")\n    .agg(\n        total_frames=("frame_idx","count"),\n        localized_frames=("stage1_detected","sum"),\n        strict_FA=("correct_strict","mean"),\n        operational_FA_with_nothing_policy=("correct_operational","mean"),\n        localization_failure_rate=("localization_failure","mean"),\n        recognition_failure_rate_all_frames=("recognition_failure","mean"),\n        mean_stage1_latency_ms=("stage1_latency_ms","mean"),\n        mean_model_call_ms=("model_call_ms","mean"),\n        mean_component_sum_e2e_ms=("component_sum_e2e_ms","mean"),\n    )\n    .reset_index()\n)\nper_video.to_csv(out_dir/"per_video_summary.csv",index=False)\n\nsummary={\n    "model":MODEL_NAME,\n    "branch":"classification",\n    "protocol":"shared_YOLOv8_stage1_then_classifier",\n    "effnet_load_mode":(\n        "kaggle_compat_custom_layer_equivalent_preprocess"\n        if MODEL_NAME=="EfficientNetB0"\n        else None\n    ),\n    "effnet_compatibility_note":(\n        "Serialized no-weight effnet_preprocess Lambda replaced by "\n        "EffNetPreprocessCompat implementing preprocess_input(x*255.0); "\n        "checkpoint weights unchanged."\n        if MODEL_NAME=="EfficientNetB0"\n        else None\n    ),\n    "n_frames":int(len(df)),\n    "label_alias_policy":"del->delete; single letters upper-cased; delete/nothing/space lower-cased",\n    "label_alias_applied_frames":int(df["label_alias_applied"].sum()),\n    "vit_preprocessing":(\n        "Resize(224,224)->ToTensor()->Normalize(mean=[0.5]*3,std=[0.5]*3)"\n        if MODEL_NAME=="ViT" else None\n    ),\n    "strict_FA_all_frames":float(df["correct_strict"].mean()),\n    "operational_FA_with_nothing_policy":float(df["correct_operational"].mean()),\n    "conditional_accuracy_given_localization":float(localized["correct_strict"].mean()) if len(localized) else None,\n    "localization_failure_rate":float(df["localization_failure"].mean()),\n    "recognition_failure_rate_all_frames":float(df["recognition_failure"].mean()),\n    "recognition_failure_rate_given_localization":float(localized["recognition_failure"].mean()) if len(localized) else None,\n    "TS_source":"shared_YOLOv8_stage1",\n    "TS_detected_pairs":shared_ts.get("TS_detected_pairs"),\n    "TS_zero_missing":shared_ts.get("TS_zero_missing"),\n    "mean_stage1_latency_ms":float(df["stage1_latency_ms"].mean()),\n    "mean_roi_crop_ms":float(localized["roi_crop_ms"].mean()) if len(localized) else None,\n    "mean_preprocess_ms":float(localized["preprocess_ms"].mean()) if len(localized) else None,\n    "mean_model_call_ms":float(localized["model_call_ms"].mean()) if len(localized) else None,\n    "mean_postprocess_ms":float(localized["postprocess_ms"].mean()) if len(localized) else None,\n    "model_only_FPS":float(1000.0/localized["model_call_ms"].mean()) if len(localized) and localized["model_call_ms"].mean()>0 else None,\n    "component_sum_e2e_ms":float(df["component_sum_e2e_ms"].mean()),\n    "component_sum_e2e_FPS_estimate":float(1000.0/df["component_sum_e2e_ms"].mean()),\n    "FPS_note":(\n        "model_only_FPS uses target model-call latency only. "\n        "component_sum_e2e_FPS_estimate sums measured Stage-1, ROI crop, "\n        "preprocess, target model-call, and postprocess latencies; video decoding "\n        "and single-process integrated wall-clock overhead are not included."\n    ),\n    "nothing_policy_note":(\n        "Strict FA treats every missing Stage-1 localization as failure. "\n        "Operational FA additionally interprets a Stage-1 no-hand result as "\n        "the system output \'nothing\' only when the ground-truth class is \'nothing\'."\n    ),\n    "nothing_frames":int((df["true_label"]=="nothing").sum()),\n    "nothing_frames_no_stage1_detection":int(\n        ((df["true_label"]=="nothing") & (~df["stage1_detected"])).sum()\n    ),\n    "nothing_operational_accuracy":(\n        float(df.loc[df["true_label"]=="nothing","correct_operational"].mean())\n        if (df["true_label"]=="nothing").any()\n        else None\n    ),\n}\n\n(out_dir/"metrics_summary.json").write_text(\n    json.dumps(summary,indent=2),\n    encoding="utf-8"\n)\n\nprint("MODEL COMPLETE",MODEL_NAME)\n',
    encoding="utf-8"
)

DET_WORKER.write_text(
    '\nimport sys,json,time\nfrom pathlib import Path\n\nMODEL_NAME=sys.argv[1]\nMODEL_PATH=Path(sys.argv[2])\nSTAGE1_CSV=Path(sys.argv[3])\nVIDEOS=[Path(x) for x in json.loads(sys.argv[4])]\nOUT_ROOT=Path(sys.argv[5])\n\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport torch\n\nfrom ultralytics import YOLO\ntry:\n    from ultralytics import RTDETR\nexcept Exception:\n    RTDETR=None\n\nif MODEL_NAME=="RTDETR_L" and RTDETR is not None:\n    try:\n        model=RTDETR(str(MODEL_PATH))\n    except Exception:\n        model=YOLO(str(MODEL_PATH))\nelse:\n    model=YOLO(str(MODEL_PATH))\n\nstage1=pd.read_csv(STAGE1_CSV)\nlookup={(r.video,int(r.frame_idx)):r for r in stage1.itertuples(index=False)}\n\nout_dir=OUT_ROOT/"detectors_two_stage"/MODEL_NAME\nparts_dir=out_dir/"per_video"\nparts_dir.mkdir(parents=True,exist_ok=True)\n\ndef normalize_label(x):\n    s=str(x).strip(); low=s.lower()\n    if low=="del":\n        return "delete"\n    if low in {"delete","nothing","space"}:\n        return low\n    if len(s)==1:\n        return s.upper()\n    return s\n\ndef box_iou(a,b):\n    if a is None or b is None:\n        return None\n    xA=max(a[0],b[0]); yA=max(a[1],b[1])\n    xB=min(a[2],b[2]); yB=min(a[3],b[3])\n    inter=max(0.0,xB-xA)*max(0.0,yB-yA)\n    aa=max(0.0,a[2]-a[0])*max(0.0,a[3]-a[1])\n    ab=max(0.0,b[2]-b[0])*max(0.0,b[3]-b[1])\n    den=aa+ab-inter\n    return inter/den if den>0 else 0.0\n\ndef infer(roi):\n    p0=time.perf_counter()\n    resized=cv2.resize(roi,(224,224))\n    p1=time.perf_counter()\n\n    torch.cuda.synchronize()\n    m0=time.perf_counter()\n\n    r=model.predict(\n        resized,\n        imgsz=224,\n        verbose=False,\n        device=0\n    )[0]\n\n    torch.cuda.synchronize()\n    m1=time.perf_counter()\n\n    post0=time.perf_counter()\n\n    if r.boxes is None or len(r.boxes)==0:\n        post1=time.perf_counter()\n        return "__NO_DETECTION__","__NO_DETECTION__",None,None,(p1-p0)*1000.0,(m1-m0)*1000.0,(post1-post0)*1000.0\n\n    confs=r.boxes.conf.detach().cpu().numpy()\n    boxes=r.boxes.xyxy.detach().cpu().numpy()\n    clss=r.boxes.cls.detach().cpu().numpy().astype(int)\n\n    j=int(np.argmax(confs))\n    cls_i=int(clss[j])\n\n    name=model.names[cls_i] if isinstance(model.names,dict) else model.names[cls_i]\n    raw_pred=str(name)\n    pred=normalize_label(raw_pred)\n    conf=float(confs[j])\n    box=boxes[j].astype(float).tolist()\n\n    post1=time.perf_counter()\n\n    return raw_pred,pred,conf,box,(p1-p0)*1000.0,(m1-m0)*1000.0,(post1-post0)*1000.0\n\nfor vp in VIDEOS:\n    final_csv=parts_dir/f"{vp.stem}.csv"\n\n    if final_csv.exists():\n        # Reuse only outputs created with the NOTHING-aware schema.\n        try:\n            cached_cols = set(pd.read_csv(final_csv, nrows=1).columns)\n        except Exception:\n            cached_cols = set()\n\n        required_cols = {\n            "pred_label_operational",\n            "correct_operational",\n            "operational_nothing_policy_applied",\n        }\n\n        if required_cols.issubset(cached_cols):\n            print("SKIP",vp.name)\n            continue\n\n        print("REBUILD old-schema cache:",vp.name)\n        final_csv.unlink()\n\n    gt=normalize_label(vp.stem)\n    cap=cv2.VideoCapture(str(vp))\n\n    idx=-1\n    rows=[]\n    prev_target_box=None\n\n    while True:\n        ok,frame=cap.read()\n        if not ok:\n            break\n\n        idx+=1\n        s=lookup.get((vp.name,idx))\n        if s is None:\n            continue\n\n        target_box=None\n        target_pair=None\n\n        if not bool(s.detected):\n            strict_pred="__NO_HAND__"\n            strict_correct=False\n\n            operational_policy_applied=(gt=="nothing")\n            operational_pred=(\n                "nothing"\n                if operational_policy_applied\n                else "__NO_HAND__"\n            )\n            operational_correct=(\n                operational_pred==gt\n            )\n\n            rows.append({\n                "video":vp.name,\n                "frame_idx":idx,\n                "true_label":gt,\n                "stage1_detected":False,\n\n                "pred_label_raw":strict_pred,\n                "pred_label":strict_pred,\n                "label_alias_applied":False,\n                "pred_conf":np.nan,\n                "correct_strict":strict_correct,\n\n                "pred_label_operational":operational_pred,\n                "correct_operational":operational_correct,\n                "operational_nothing_policy_applied":operational_policy_applied,\n\n                "localization_failure":True,\n                "target_detection_failure":False,\n                "recognition_failure":False,\n                "target_x1":np.nan,"target_y1":np.nan,\n                "target_x2":np.nan,"target_y2":np.nan,\n                "target_iou_with_prev_detected_pair":np.nan,\n                "target_iou_with_prev_zero_missing":0.0 if idx>0 else np.nan,\n                "stage1_latency_ms":float(s.stage1_latency_ms),\n                "roi_crop_ms":0.0,\n                "target_preprocess_ms":0.0,\n                "target_model_call_ms":0.0,\n                "target_postprocess_ms":0.0,\n                "component_sum_e2e_ms":float(s.stage1_latency_ms),\n            })\n            prev_target_box=None\n            continue\n\n        x1,y1,x2,y2=[int(round(v)) for v in [s.x1,s.y1,s.x2,s.y2]]\n        h,w=frame.shape[:2]\n\n        x1=max(0,min(x1,w-1)); x2=max(1,min(x2,w))\n        y1=max(0,min(y1,h-1)); y2=max(1,min(y2,h))\n\n        c0=time.perf_counter()\n        roi=frame[y1:y2,x1:x2]\n        c1=time.perf_counter()\n        crop_ms=(c1-c0)*1000.0\n\n        if roi.size==0:\n            raw_pred="__EMPTY_ROI__"\n            pred="__EMPTY_ROI__"\n            conf=np.nan\n            local_box=None\n            pre_ms=model_ms=post_ms=0.0\n        else:\n            raw_pred,pred,conf,local_box,pre_ms,model_ms,post_ms=infer(roi)\n\n        if local_box is not None:\n            # The target detector runs on a 224x224 resized ROI. Convert its\n            # local box back to the original ROI coordinate system before\n            # mapping to full-frame coordinates.\n            roi_h,roi_w=roi.shape[:2]\n            sx=float(roi_w)/224.0\n            sy=float(roi_h)/224.0\n            target_box=[\n                float(local_box[0]*sx+x1),\n                float(local_box[1]*sy+y1),\n                float(local_box[2]*sx+x1),\n                float(local_box[3]*sy+y1),\n            ]\n\n        target_pair=(\n            box_iou(prev_target_box,target_box)\n            if prev_target_box is not None and target_box is not None\n            else None\n        )\n\n        target_detection_failure=(pred=="__NO_DETECTION__")\n        correct=pred==gt\n\n        rows.append({\n            "video":vp.name,\n            "frame_idx":idx,\n            "true_label":gt,\n            "stage1_detected":True,\n            "pred_label_raw":raw_pred,\n            "pred_label":pred,\n            "label_alias_applied":bool(str(raw_pred).strip().lower()=="del" and pred=="delete"),\n            "pred_conf":conf,\n            "correct_strict":correct,\n            "pred_label_operational":pred,\n            "correct_operational":correct,\n            "operational_nothing_policy_applied":False,\n            "localization_failure":False,\n            "target_detection_failure":target_detection_failure,\n            "recognition_failure":(not correct) and (not target_detection_failure),\n            "target_x1":target_box[0] if target_box else np.nan,\n            "target_y1":target_box[1] if target_box else np.nan,\n            "target_x2":target_box[2] if target_box else np.nan,\n            "target_y2":target_box[3] if target_box else np.nan,\n            "target_iou_with_prev_detected_pair":target_pair,\n            "target_iou_with_prev_zero_missing":(\n                target_pair if target_pair is not None else (0.0 if idx>0 else np.nan)\n            ),\n            "stage1_latency_ms":float(s.stage1_latency_ms),\n            "roi_crop_ms":crop_ms,\n            "target_preprocess_ms":pre_ms,\n            "target_model_call_ms":model_ms,\n            "target_postprocess_ms":post_ms,\n            "component_sum_e2e_ms":(\n                float(s.stage1_latency_ms)\n                + crop_ms+pre_ms+model_ms+post_ms\n            ),\n        })\n\n        prev_target_box=target_box\n\n    cap.release()\n\n    tmp=final_csv.with_suffix(".tmp.csv")\n    pd.DataFrame(rows).to_csv(tmp,index=False)\n    tmp.replace(final_csv)\n\n    print("DONE",MODEL_NAME,vp.name)\n\ndf=pd.concat(\n    [pd.read_csv(parts_dir/f"{vp.stem}.csv") for vp in VIDEOS],\n    ignore_index=True\n)\n\ndf.to_csv(out_dir/"frame_level_outputs.csv",index=False)\n\nlocalized=df[df["stage1_detected"]==True]\ntarget_ts=df["target_iou_with_prev_detected_pair"].dropna()\ntarget_ts_zero=df["target_iou_with_prev_zero_missing"].dropna()\n\nper_video=(\n    df.groupby("video")\n    .agg(\n        total_frames=("frame_idx","count"),\n        localized_frames=("stage1_detected","sum"),\n        strict_FA=("correct_strict","mean"),\n        operational_FA_with_nothing_policy=("correct_operational","mean"),\n        localization_failure_rate=("localization_failure","mean"),\n        target_detection_failure_rate=("target_detection_failure","mean"),\n        recognition_failure_rate=("recognition_failure","mean"),\n        TS_target_detected_pairs=("target_iou_with_prev_detected_pair","mean"),\n        TS_target_zero_missing=("target_iou_with_prev_zero_missing","mean"),\n        mean_target_model_call_ms=("target_model_call_ms","mean"),\n        mean_component_sum_e2e_ms=("component_sum_e2e_ms","mean"),\n    )\n    .reset_index()\n)\nper_video.to_csv(out_dir/"per_video_summary.csv",index=False)\n\nsummary={\n    "model":MODEL_NAME,\n    "branch":"detection",\n    "protocol":"postpublication_two_stage_YOLOv8_ROI_then_target_detector_as_described_in_Section_4_3",\n    "n_frames":int(len(df)),\n    "label_alias_policy":"del->delete; single letters upper-cased; delete/nothing/space lower-cased",\n    "label_alias_applied_frames":int(df["label_alias_applied"].sum()),\n    "target_box_mapping":"224x224 target-detector ROI boxes scaled back to original ROI size, then offset to full-frame coordinates",\n    "strict_FA_all_frames":float(df["correct_strict"].mean()),\n    "operational_FA_with_nothing_policy":float(df["correct_operational"].mean()),\n    "conditional_accuracy_given_stage1_localization":float(localized["correct_strict"].mean()) if len(localized) else None,\n    "localization_failure_rate":float(df["localization_failure"].mean()),\n    "target_detection_failure_rate_given_stage1":float(localized["target_detection_failure"].mean()) if len(localized) else None,\n    "recognition_failure_rate_given_stage1":float(localized["recognition_failure"].mean()) if len(localized) else None,\n    "TS_source":"target_detector_boxes_mapped_back_to_full_frame",\n    "TS_detected_pairs":float(target_ts.mean()) if len(target_ts) else None,\n    "TS_zero_missing":float(target_ts_zero.mean()) if len(target_ts_zero) else None,\n    "mean_stage1_latency_ms":float(df["stage1_latency_ms"].mean()),\n    "mean_target_model_call_ms":float(localized["target_model_call_ms"].mean()) if len(localized) else None,\n    "model_only_FPS":float(1000.0/localized["target_model_call_ms"].mean()) if len(localized) and localized["target_model_call_ms"].mean()>0 else None,\n    "component_sum_e2e_ms":float(df["component_sum_e2e_ms"].mean()),\n    "component_sum_e2e_FPS_estimate":float(1000.0/df["component_sum_e2e_ms"].mean()),\n    "FPS_note":"model_only_FPS uses target model-call latency only. component_sum_e2e_FPS_estimate sums measured Stage-1, ROI crop, target preprocess, target model-call, and postprocess latencies; video decoding and single-process integrated wall-clock overhead are not included.",\n    "nothing_policy_note":(\n        "Strict FA treats missing Stage-1 localization as failure. "\n        "Operational FA maps a no-hand Stage-1 result to \'nothing\' only "\n        "for ground-truth \'nothing\'."\n    ),\n    "nothing_frames":int((df["true_label"]=="nothing").sum()),\n    "nothing_frames_no_stage1_detection":int(\n        ((df["true_label"]=="nothing") & (~df["stage1_detected"])).sum()\n    ),\n    "nothing_operational_accuracy":(\n        float(df.loc[df["true_label"]=="nothing","correct_operational"].mean())\n        if (df["true_label"]=="nothing").any()\n        else None\n    ),\n}\n\n(out_dir/"metrics_summary.json").write_text(\n    json.dumps(summary,indent=2),\n    encoding="utf-8"\n)\n\nprint("MODEL COMPLETE",MODEL_NAME)\n',
    encoding="utf-8"
)

print("Workers ready.")
print("EfficientNet worker: verified Keras-3 mask/training compatibility fix.")


## Shared Stage-1 YOLOv8 hand localization

Stage 1 is executed once in an isolated process on one Tesla T4. Its frame-level bounding boxes, confidence values, failure cases, latency, and consecutive-frame IoU values are persisted and reused by every downstream model.


In [ ]:
# ============================================================
# 5. RUN STAGE-1 SEQUENTIALLY
# ============================================================

STAGE1_DIR = (
    WORK_ROOT
    / "stage1_yolov8"
)

STAGE1_PARTS = (
    STAGE1_DIR
    / "per_video"
)

STAGE1_PARTS.mkdir(
    parents=True,
    exist_ok=True
)

video_json = json.dumps(
    [
        str(v)
        for v in VIDEOS
    ]
)

stage1_cmd = [
    sys.executable,
    str(
        STAGE1_WORKER
    ),
    str(
        HAND_DETECTOR_PATH
    ),
    str(
        STAGE1_PARTS
    ),
    str(
        HAND_CONF
    ),
    str(
        HAND_IMGSZ
    ),
    str(
        FRAME_STRIDE
    ),
    video_json,
]

stage1_run = run_sequential_child(
    "YOLOv8_Stage1",
    stage1_cmd,
    timeout=6*60*60
)

# If worker stopped, rerun only missing videos one-by-one.
missing = [
    v
    for v in VIDEOS
    if not (
        STAGE1_PARTS
        / f"{v.stem}.csv"
    ).exists()
]

if missing:
    print(
        "Retry missing Stage-1 videos:",
        [
            v.name
            for v in missing
        ]
    )

    for v in missing:
        retry_cmd = [
            sys.executable,
            str(
                STAGE1_WORKER
            ),
            str(
                HAND_DETECTOR_PATH
            ),
            str(
                STAGE1_PARTS
            ),
            str(
                HAND_CONF
            ),
            str(
                HAND_IMGSZ
            ),
            str(
                FRAME_STRIDE
            ),
            json.dumps(
                [
                    str(v)
                ]
            ),
        ]

        rr = run_sequential_child(
            f"Stage1::{v.name}",
            retry_cmd,
            timeout=90*60
        )

        if rr[
            "returncode"
        ] != 0:
            raise RuntimeError(
                f"Stage-1 failed on {v.name}"
            )

missing = [
    v
    for v in VIDEOS
    if not (
        STAGE1_PARTS
        / f"{v.stem}.csv"
    ).exists()
]

if missing:
    raise RuntimeError(
        "Stage-1 incomplete."
    )

print(
    "✅ Stage-1 complete."
)


In [ ]:
# ============================================================
# 6. STAGE-1 SUMMARY
# ============================================================

stage1_df = pd.concat(
    [
        pd.read_csv(
            STAGE1_PARTS
            / f"{v.stem}.csv"
        )
        for v in VIDEOS
    ],
    ignore_index=True
)

STAGE1_CSV = (
    STAGE1_DIR
    / "frame_level_stage1.csv"
)

stage1_df.to_csv(
    STAGE1_CSV,
    index=False
)

valid_ts = (
    stage1_df[
        "iou_with_prev_detected_pair"
    ]
    .dropna()
)

zero_ts = (
    stage1_df[
        "iou_with_prev_zero_missing"
    ]
    .dropna()
)

stage1_summary = {
    "component":
        "YOLOv8_Stage1",

    "status":
        "PASS",

    "total_videos":
        int(
            len(
                VIDEOS
            )
        ),

    "total_frames":
        int(
            len(
                stage1_df
            )
        ),

    "detected_frames":
        int(
            stage1_df[
                "detected"
            ].sum()
        ),

    "no_detection_frames":
        int(
            (
                ~stage1_df[
                    "detected"
                ]
            ).sum()
        ),

    "multiple_detection_frames":
        int(
            stage1_df[
                "multiple_detections"
            ].sum()
        ),

    "detection_success_rate":
        float(
            stage1_df[
                "detected"
            ].mean()
        ),

    "localization_failure_rate":
        float(
            (
                ~stage1_df[
                    "detected"
                ]
            ).mean()
        ),

    "TS_detected_pairs":
        (
            float(
                valid_ts.mean()
            )
            if len(
                valid_ts
            )
            else None
        ),

    "TS_zero_missing":
        (
            float(
                zero_ts.mean()
            )
            if len(
                zero_ts
            )
            else None
        ),

    "mean_stage1_latency_ms":
        float(
            stage1_df[
                "stage1_latency_ms"
            ].mean()
        ),

    "stage1_FPS":
        float(
            1000.0
            /
            stage1_df[
                "stage1_latency_ms"
            ].mean()
        ),

    "confidence_threshold":
        HAND_CONF,

    "imgsz":
        HAND_IMGSZ,

    "weights_path":
        str(
            HAND_DETECTOR_PATH
        ),

    "weights_sha256":
        sha256_file(
            HAND_DETECTOR_PATH
        ),

    "weight_origin":
        "NOT_ESTABLISHED_FROM_AVAILABLE_CHECKPOINT_METADATA",

    "gpu":
        TEST_GPU_NAME,
}

save_json(
    stage1_summary,
    STAGE1_DIR
    / "stage1_summary.json"
)

display(
    pd.DataFrame(
        [
            stage1_summary
        ]
    )
)


## Five classification models — sequential evaluation

Models are evaluated one at a time:

`Custom CNN → EfficientNetB0 → ResNet50V2 → ViT → YOLOv11m-cls`

Each model runs in its own subprocess so CUDA/framework state is released before the next model. Classification TS is inherited from the shared Stage-1 boxes and is therefore not interpreted as an intrinsic property of the classifier.


In [ ]:
# ============================================================
# 7. RUN 5 CLASSIFIERS SEQUENTIALLY
# ============================================================

shared_ts_json = (
    STAGE1_DIR
    / "stage1_summary.json"
)

classifier_run_status = []

for model_name, original_path in CLASSIFICATION_MODELS:

    print(
        "\n\n########## CLASSIFIER:",
        model_name,
        "##########"
    )

    if model_name == "EfficientNetB0":
        model_path = EFF_COMPAT
        effnet_load_mode = (
            "kaggle_compat_custom_layer_equivalent_preprocess"
        )
    else:
        model_path = original_path
        effnet_load_mode = None

    cmd = [
        sys.executable,
        str(
            CLS_WORKER
        ),
        model_name,
        str(
            model_path
        ),
        str(
            STAGE1_CSV
        ),
        video_json,
        str(
            WORK_ROOT
        ),
        str(
            shared_ts_json
        ),
    ]

    result = run_sequential_child(
        f"classifier::{model_name}",
        cmd,
        timeout=8*60*60
    )

    classifier_run_status.append({
        "model":
            model_name,

        "returncode":
            result[
                "returncode"
            ],

        "wall_seconds":
            result[
                "wall_seconds"
            ],

        "effnet_load_mode":
            effnet_load_mode,
    })

    if result[
        "returncode"
    ] != 0:

        print(
            f"❌ {model_name} failed."
        )

        if model_name == "EfficientNetB0":
            print(
                "EfficientNet compatibility reconstruction failed. "
                "The failure is retained as evidence; no further fallback is used."
            )

        print(
            "Continue with remaining models."
        )


df_classifier_status = pd.DataFrame(
    classifier_run_status
)

display(
    df_classifier_status
)


## Three detection models — sequential two-stage re-evaluation

The post-publication re-evaluation implements the two-stage pipeline described in published Section 4.3:

`video frame → YOLOv8 Stage-1 → hand ROI → target detector`

This notebook does not claim that historical execution provenance has been independently recovered unless separate archived logs are available. Target-detector boxes predicted on the resized 224×224 ROI are rescaled to the original ROI before full-frame TS is computed.


In [ ]:
# ============================================================
# 8. RUN 3 DETECTORS SEQUENTIALLY
# ============================================================

detector_run_status = []

for model_name, model_path in DETECTION_MODELS:

    print(
        "\n\n########## DETECTOR:",
        model_name,
        "##########"
    )

    cmd = [
        sys.executable,
        str(
            DET_WORKER
        ),
        model_name,
        str(
            model_path
        ),
        str(
            STAGE1_CSV
        ),
        video_json,
        str(
            WORK_ROOT
        ),
    ]

    result = run_sequential_child(
        f"detector::{model_name}",
        cmd,
        timeout=10*60*60
    )

    detector_run_status.append({
        "model":
            model_name,

        "returncode":
            result[
                "returncode"
            ],

        "wall_seconds":
            result[
                "wall_seconds"
            ],
    })

    if result[
        "returncode"
    ] != 0:
        print(
            f"❌ {model_name} failed. "
            "Continuing so the final summary remains complete."
        )


df_detector_status = pd.DataFrame(
    detector_run_status
)

display(
    df_detector_status
)


# Final editor summary

The summary contains the shared Stage-1 row plus all eight target models and records:

- strict and operational FA;
- localization and recognition failure rates;
- TS value and explicit TS source;
- target-model-only FPS;
- measured component-sum pipeline FPS estimate;
- Stage-1 settings and checkpoint hash;
- published Table 5 values for side-by-side audit;
- ViT preprocessing provenance;
- label-alias application counts;
- target-detector box-remapping method;
- checkpoint semantic-label audit status.

The final verification cell then packages all frame-level outputs, manifests, logs, summaries, and audit files into one ZIP for editorial review.


In [ ]:
# ============================================================
# 9. BUILD THE SINGLE FINAL EDITOR SUMMARY FILE
# ============================================================

PUBLISHED_TABLE5 = {
    "Custom_CNN":{
        "FA_pct":3.88,
        "TS_pct":94.75,
        "FPS":11.21,
    },

    "ResNet50V2":{
        "FA_pct":12.02,
        "TS_pct":86.97,
        "FPS":10.62,
    },

    "EfficientNetB0":{
        "FA_pct":47.00,
        "TS_pct":87.80,
        "FPS":11.20,
    },

    "YOLOv11m_cls":{
        "FA_pct":26.74,
        "TS_pct":88.82,
        "FPS":54.96,
    },

    "ViT":{
        "FA_pct":81.93,
        "TS_pct":94.78,
        "FPS":2.34,
    },

    "YOLOv11m_det":{
        "FA_pct":27.91,
        "TS_pct":80.00,
        "FPS":43.10,
    },

    "RTDETR_L":{
        "FA_pct":48.82,
        "TS_pct":90.72,
        "FPS":1.73,
    },

    "YOLOv12x":{
        "FA_pct":43.19,
        "TS_pct":85.99,
        "FPS":25.82,
    },
}

rows = []

checkpoint_sha_by_model = {
    str(r.model): str(r.checkpoint_sha256)
    for r in model_manifest_df.itertuples(index=False)
}

label_audit_map = {}
if LABEL_AUDIT_CSV.exists():
    _ladf = pd.read_csv(LABEL_AUDIT_CSV)
    label_audit_map = {
        str(r.model): r
        for r in _ladf.itertuples(index=False)
    }


# ------------------------------------------------------------
# Shared Stage-1 row
# ------------------------------------------------------------
rows.append({
    "model":
        "YOLOv8_Stage1",

    "evaluation_version":
        EVALUATION_VERSION,

    "target_checkpoint_sha256":
        checkpoint_sha_by_model.get("YOLOv8_Stage1"),

    "branch":
        "shared_localization",

    "status":
        "PASS",

    "protocol":
        "full_frame_to_YOLOv8_hand_localization",

    "total_videos":
        stage1_summary[
            "total_videos"
        ],

    "total_frames":
        stage1_summary[
            "total_frames"
        ],

    "strict_FA_pct":
        None,

    "operational_FA_nothing_pct":
        None,

    "conditional_accuracy_given_localization_pct":
        None,

    "localization_failure_pct":
        100.0
        * stage1_summary[
            "localization_failure_rate"
        ],

    "recognition_failure_given_localization_pct":
        None,

    "target_detection_failure_given_localization_pct":
        None,

    "TS_detected_pairs_pct":
        (
            100.0
            * stage1_summary[
                "TS_detected_pairs"
            ]
            if stage1_summary[
                "TS_detected_pairs"
            ] is not None
            else None
        ),

    "TS_zero_missing_pct":
        (
            100.0
            * stage1_summary[
                "TS_zero_missing"
            ]
            if stage1_summary[
                "TS_zero_missing"
            ] is not None
            else None
        ),

    "TS_source":
        "YOLOv8_Stage1_boxes",

    "mean_stage1_latency_ms":
        stage1_summary[
            "mean_stage1_latency_ms"
        ],

    "mean_target_latency_ms":
        None,

    "model_only_FPS":
        stage1_summary[
            "stage1_FPS"
        ],

    "component_sum_e2e_ms":
        None,

    "component_sum_e2e_FPS_estimate":
        None,

    "nothing_frames":
        None,

    "nothing_frames_no_stage1_detection":
        None,

    "nothing_operational_accuracy_pct":
        None,

    "stage1_detection_success_pct":
        100.0
        * stage1_summary[
            "detection_success_rate"
        ],

    "stage1_no_detection_frames":
        stage1_summary[
            "no_detection_frames"
        ],

    "stage1_multiple_detection_frames":
        stage1_summary[
            "multiple_detection_frames"
        ],

    "stage1_conf_threshold":
        HAND_CONF,

    "stage1_imgsz":
        HAND_IMGSZ,

    "stage1_weights_sha256":
        stage1_summary[
            "weights_sha256"
        ],

    "stage1_weight_origin":
        stage1_summary[
            "weight_origin"
        ],

    "gpu":
        TEST_GPU_NAME,

    "effnet_load_mode":
        None,

    "effnet_compatibility_note":
        None,

    "published_FA_pct":
        None,

    "published_TS_pct":
        None,

    "published_FPS":
        None,

    "FA_diff_rerun_minus_published_pp":
        None,

    "TS_diff_rerun_minus_published_pp":
        None,

    "label_alias_applied_frames":
        None,

    "vit_preprocessing":
        None,

    "target_box_mapping":
        None,

    "ultralytics_semantic_label_set_match":
        None,

    "timing_scope":
        "Stage-1 model.predict latency only",

    "exact_single_process_wallclock_e2e_measured":
        False,

    "notes":
        (
            "Shared localization component. "
            "TS_detected_pairs is undefined for an individual video "
            "when no adjacent detected bbox pair exists."
        ),
})

# ------------------------------------------------------------
# Helper for status lookup
# ------------------------------------------------------------
cls_status_map = {
    str(r["model"]):
        r
    for r in classifier_run_status
}

det_status_map = {
    str(r["model"]):
        r
    for r in detector_run_status
}


def pct(x):
    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    return 100.0 * float(x)


# ------------------------------------------------------------
# 5 classifiers
# ------------------------------------------------------------
for model_name, _ in CLASSIFICATION_MODELS:

    metric_path = (
        WORK_ROOT
        / "classifiers"
        / model_name
        / "metrics_summary.json"
    )

    status_info = cls_status_map.get(
        model_name,
        {}
    )

    if metric_path.exists():
        m = json.loads(
            metric_path.read_text()
        )

        status = (
            "PASS"
        )

    else:
        m = {}

        status = (
            "FAILED"
        )

    pub = PUBLISHED_TABLE5[
        model_name
    ]

    rerun_fa = pct(
        m.get(
            "strict_FA_all_frames"
        )
    )

    rerun_ts = pct(
        m.get(
            "TS_detected_pairs"
        )
    )

    rows.append({
        "model":
            model_name,

        "evaluation_version":
            EVALUATION_VERSION,

        "target_checkpoint_sha256":
            checkpoint_sha_by_model.get(model_name),

        "branch":
            "classification",

        "status":
            status,

        "protocol":
            m.get(
                "protocol",
                "shared_YOLOv8_stage1_then_classifier"
            ),

        "total_videos":
            len(
                VIDEOS
            ),

        "total_frames":
            m.get(
                "n_frames"
            ),

        "strict_FA_pct":
            rerun_fa,

        "operational_FA_nothing_pct":
            pct(
                m.get(
                    "operational_FA_with_nothing_policy"
                )
            ),

        "conditional_accuracy_given_localization_pct":
            pct(
                m.get(
                    "conditional_accuracy_given_localization"
                )
            ),

        "localization_failure_pct":
            pct(
                m.get(
                    "localization_failure_rate"
                )
            ),

        "recognition_failure_given_localization_pct":
            pct(
                m.get(
                    "recognition_failure_rate_given_localization"
                )
            ),

        "target_detection_failure_given_localization_pct":
            None,

        "TS_detected_pairs_pct":
            rerun_ts,

        "TS_zero_missing_pct":
            pct(
                m.get(
                    "TS_zero_missing"
                )
            ),

        "TS_source":
            m.get(
                "TS_source",
                "shared_YOLOv8_stage1"
            ),

        "mean_stage1_latency_ms":
            m.get(
                "mean_stage1_latency_ms"
            ),

        "mean_target_latency_ms":
            m.get(
                "mean_model_call_ms"
            ),

        "model_only_FPS":
            m.get(
                "model_only_FPS"
            ),

        "component_sum_e2e_ms":
            m.get(
                "component_sum_e2e_ms"
            ),

        "component_sum_e2e_FPS_estimate":
            m.get(
                "component_sum_e2e_FPS_estimate"
            ),

        "nothing_frames":
            m.get(
                "nothing_frames"
            ),

        "nothing_frames_no_stage1_detection":
            m.get(
                "nothing_frames_no_stage1_detection"
            ),

        "nothing_operational_accuracy_pct":
            pct(
                m.get(
                    "nothing_operational_accuracy"
                )
            ),

        "stage1_detection_success_pct":
            100.0
            * stage1_summary[
                "detection_success_rate"
            ],

        "stage1_no_detection_frames":
            stage1_summary[
                "no_detection_frames"
            ],

        "stage1_multiple_detection_frames":
            stage1_summary[
                "multiple_detection_frames"
            ],

        "stage1_conf_threshold":
            HAND_CONF,

        "stage1_imgsz":
            HAND_IMGSZ,

        "stage1_weights_sha256":
            stage1_summary[
                "weights_sha256"
            ],

        "stage1_weight_origin":
            stage1_summary[
                "weight_origin"
            ],

        "gpu":
            TEST_GPU_NAME,

        "effnet_load_mode":
            (
                m.get(
                    "effnet_load_mode"
                )
                or
                status_info.get(
                    "effnet_load_mode"
                )
            ),

        "effnet_compatibility_note":
            m.get(
                "effnet_compatibility_note"
            ),

        "label_alias_applied_frames":
            m.get(
                "label_alias_applied_frames"
            ),

        "vit_preprocessing":
            m.get(
                "vit_preprocessing"
            ),

        "target_box_mapping":
            None,

        "ultralytics_semantic_label_set_match":
            (
                bool(
                    getattr(
                        label_audit_map.get(model_name),
                        "semantic_label_set_match",
                        False
                    )
                )
                if model_name=="YOLOv11m_cls"
                else None
            ),

        "timing_scope":
            (
                "model_only_FPS = Stage-2 model-call only; "
                "component_sum_e2e_FPS_estimate = measured Stage-1 + crop + "
                "preprocess + Stage-2 model-call + postprocess; video decode and "
                "integrated single-process overhead excluded"
            ),

        "exact_single_process_wallclock_e2e_measured":
            False,

        "published_FA_pct":
            pub[
                "FA_pct"
            ],

        "published_TS_pct":
            pub[
                "TS_pct"
            ],

        "published_FPS":
            pub[
                "FPS"
            ],

        "FA_diff_rerun_minus_published_pp":
            (
                rerun_fa
                -
                pub[
                    "FA_pct"
                ]
                if rerun_fa
                is not None
                else None
            ),

        "TS_diff_rerun_minus_published_pp":
            (
                rerun_ts
                -
                pub[
                    "TS_pct"
                ]
                if rerun_ts
                is not None
                else None
            ),

        "notes":
            (
                "Classification TS is inherited from the shared YOLOv8 Stage-1 "
                "and is not an intrinsic classifier property. "
                "Strict FA is the primary reproduction metric; "
                "operational NOTHING-policy FA is supplemental. "
                "For EfficientNetB0 only, the old serialized no-weight preprocessing "
                "Lambda is replaced by an equivalent Keras-3-compatible custom layer; "
                "weights are unchanged and this is explicitly a post-publication "
                "runtime compatibility reconstruction. "
                "component_sum_e2e_FPS_estimate is an explicitly labeled "
                "sum of measured sequential components, not a single-process "
                "integrated wall-clock E2E measurement."
                if status=="PASS"
                else
                f"Model evaluation failed; child returncode="
                f"{status_info.get('returncode')}."
            ),
    })


# ------------------------------------------------------------
# 3 detectors
# ------------------------------------------------------------
for model_name, _ in DETECTION_MODELS:

    metric_path = (
        WORK_ROOT
        / "detectors_two_stage"
        / model_name
        / "metrics_summary.json"
    )

    status_info = det_status_map.get(
        model_name,
        {}
    )

    if metric_path.exists():
        m = json.loads(
            metric_path.read_text()
        )

        status = (
            "PASS"
        )

    else:
        m = {}

        status = (
            "FAILED"
        )

    pub = PUBLISHED_TABLE5[
        model_name
    ]

    rerun_fa = pct(
        m.get(
            "strict_FA_all_frames"
        )
    )

    rerun_ts = pct(
        m.get(
            "TS_detected_pairs"
        )
    )

    rows.append({
        "model":
            model_name,

        "evaluation_version":
            EVALUATION_VERSION,

        "target_checkpoint_sha256":
            checkpoint_sha_by_model.get(model_name),

        "branch":
            "detection",

        "status":
            status,

        "protocol":
            m.get(
                "protocol",
                "postpublication_two_stage_YOLOv8_ROI_then_target_detector_as_described_in_Section_4_3"
            ),

        "total_videos":
            len(
                VIDEOS
            ),

        "total_frames":
            m.get(
                "n_frames"
            ),

        "strict_FA_pct":
            rerun_fa,

        "operational_FA_nothing_pct":
            pct(
                m.get(
                    "operational_FA_with_nothing_policy"
                )
            ),

        "conditional_accuracy_given_localization_pct":
            pct(
                m.get(
                    "conditional_accuracy_given_stage1_localization"
                )
            ),

        "localization_failure_pct":
            pct(
                m.get(
                    "localization_failure_rate"
                )
            ),

        "recognition_failure_given_localization_pct":
            pct(
                m.get(
                    "recognition_failure_rate_given_stage1"
                )
            ),

        "target_detection_failure_given_localization_pct":
            pct(
                m.get(
                    "target_detection_failure_rate_given_stage1"
                )
            ),

        "TS_detected_pairs_pct":
            rerun_ts,

        "TS_zero_missing_pct":
            pct(
                m.get(
                    "TS_zero_missing"
                )
            ),

        "TS_source":
            m.get(
                "TS_source"
            ),

        "mean_stage1_latency_ms":
            m.get(
                "mean_stage1_latency_ms"
            ),

        "mean_target_latency_ms":
            m.get(
                "mean_target_model_call_ms"
            ),

        "model_only_FPS":
            m.get(
                "model_only_FPS"
            ),

        "component_sum_e2e_ms":
            m.get(
                "component_sum_e2e_ms"
            ),

        "component_sum_e2e_FPS_estimate":
            m.get(
                "component_sum_e2e_FPS_estimate"
            ),

        "nothing_frames":
            m.get(
                "nothing_frames"
            ),

        "nothing_frames_no_stage1_detection":
            m.get(
                "nothing_frames_no_stage1_detection"
            ),

        "nothing_operational_accuracy_pct":
            pct(
                m.get(
                    "nothing_operational_accuracy"
                )
            ),

        "stage1_detection_success_pct":
            100.0
            * stage1_summary[
                "detection_success_rate"
            ],

        "stage1_no_detection_frames":
            stage1_summary[
                "no_detection_frames"
            ],

        "stage1_multiple_detection_frames":
            stage1_summary[
                "multiple_detection_frames"
            ],

        "stage1_conf_threshold":
            HAND_CONF,

        "stage1_imgsz":
            HAND_IMGSZ,

        "stage1_weights_sha256":
            stage1_summary[
                "weights_sha256"
            ],

        "stage1_weight_origin":
            stage1_summary[
                "weight_origin"
            ],

        "gpu":
            TEST_GPU_NAME,

        "effnet_load_mode":
            None,

        "effnet_compatibility_note":
            None,

        "label_alias_applied_frames":
            m.get(
                "label_alias_applied_frames"
            ),

        "vit_preprocessing":
            None,

        "target_box_mapping":
            m.get(
                "target_box_mapping"
            ),

        "ultralytics_semantic_label_set_match":
            bool(
                getattr(
                    label_audit_map.get(model_name),
                    "semantic_label_set_match",
                    False
                )
            ),

        "timing_scope":
            (
                "model_only_FPS = target-detector model-call only; "
                "component_sum_e2e_FPS_estimate = measured Stage-1 + crop + "
                "target preprocess + target model-call + postprocess; video decode "
                "and integrated single-process overhead excluded"
            ),

        "exact_single_process_wallclock_e2e_measured":
            False,

        "published_FA_pct":
            pub[
                "FA_pct"
            ],

        "published_TS_pct":
            pub[
                "TS_pct"
            ],

        "published_FPS":
            pub[
                "FPS"
            ],

        "FA_diff_rerun_minus_published_pp":
            (
                rerun_fa
                -
                pub[
                    "FA_pct"
                ]
                if rerun_fa
                is not None
                else None
            ),

        "TS_diff_rerun_minus_published_pp":
            (
                rerun_ts
                -
                pub[
                    "TS_pct"
                ]
                if rerun_ts
                is not None
                else None
            ),

        "notes":
            (
                "Post-publication re-evaluation implements the two-stage "
                "pipeline described in published Section 4.3. Target-detector boxes "
                "are scaled from the 224x224 target ROI back to the original ROI "
                "before mapping to full-frame coordinates for TS. Strict FA is "
                "primary; operational NOTHING-policy FA is supplemental. "
                "component_sum_e2e_FPS_estimate is an explicitly labeled component "
                "sum, not a single-process integrated wall-clock E2E measurement."
                if status=="PASS"
                else
                f"Model evaluation failed; child returncode="
                f"{status_info.get('returncode')}."
            ),
    })


final_df = pd.DataFrame(
    rows
)

# Keep a fixed ordering.
order = [
    "YOLOv8_Stage1",
    "Custom_CNN",
    "EfficientNetB0",
    "ResNet50V2",
    "ViT",
    "YOLOv11m_cls",
    "YOLOv11m_det",
    "YOLOv12x",
    "RTDETR_L",
]

final_df[
    "_order"
] = final_df[
    "model"
].map({
    name:i
    for i,name
    in enumerate(
        order
    )
})

final_df = (
    final_df
    .sort_values(
        "_order"
    )
    .drop(
        columns=[
            "_order"
        ]
    )
)

final_df.to_csv(
    FINAL_SUMMARY_CSV,
    index=False
)

display(
    final_df
)

print(
    "\n✅ FINAL EDITOR FILE:"
)

print(
    FINAL_SUMMARY_CSV
)

print(
    "\nPASS models:",
    final_df.loc[
        final_df[
            "status"
        ]=="PASS",
        "model"
    ].tolist()
)

print(
    "FAILED models:",
    final_df.loc[
        final_df[
            "status"
        ]=="FAILED",
        "model"
    ].tolist()
)


# Final verification and editor evidence package

The notebook does **not** silently convert the re-evaluation timing into an exact end-to-end FPS claim. It reports:

1. target-model-only FPS; and
2. an explicitly labeled sequential component-sum FPS estimate based on measured Stage-1, crop, preprocessing, target inference, and post-processing latencies.

Video decoding and integrated single-process wall-clock overhead are not included in the component-sum estimate.

The final verification cell checks the corrected ViT preprocessing, semantic label canonicalization, shared classification TS source, target-detector box remapping, frame-count consistency, and checkpoint-label metadata. It then creates one ZIP evidence package for editorial review.


In [ ]:
# ============================================================
# 10. FINAL VERIFICATION + EDITOR EVIDENCE PACKAGE
# ============================================================

# ------------------------------------------------------------
# Compact error-analysis table
# ------------------------------------------------------------
error_cols = [
    "model",
    "branch",
    "strict_FA_pct",
    "conditional_accuracy_given_localization_pct",
    "localization_failure_pct",
    "recognition_failure_given_localization_pct",
    "target_detection_failure_given_localization_pct",
    "TS_detected_pairs_pct",
    "TS_zero_missing_pct",
    "TS_source",
]

error_analysis_df = final_df[
    [c for c in error_cols if c in final_df.columns]
].copy()

ERROR_ANALYSIS_CSV = (
    WORK_ROOT / "IJIES_EDITOR_LOCALIZATION_RECOGNITION_ERROR_ANALYSIS.csv"
)
error_analysis_df.to_csv(ERROR_ANALYSIS_CSV, index=False)

# ------------------------------------------------------------
# Compact Table-5 re-evaluation evidence
# ------------------------------------------------------------
table5_cols = [
    "model",
    "branch",
    "strict_FA_pct",
    "operational_FA_nothing_pct",
    "TS_detected_pairs_pct",
    "TS_source",
    "model_only_FPS",
    "component_sum_e2e_FPS_estimate",
    "published_FA_pct",
    "published_TS_pct",
    "published_FPS",
    "FA_diff_rerun_minus_published_pp",
    "TS_diff_rerun_minus_published_pp",
]

table5_evidence_df = final_df[
    [c for c in table5_cols if c in final_df.columns]
].copy()

TABLE5_EVIDENCE_CSV = (
    WORK_ROOT / "IJIES_EDITOR_TABLE5_REEVALUATION_EVIDENCE.csv"
)
table5_evidence_df.to_csv(TABLE5_EVIDENCE_CSV, index=False)

# ------------------------------------------------------------
# Verification checks
# ------------------------------------------------------------
checks = {}

checks["video_count_is_29"] = (len(VIDEOS) == 29)
checks["stage1_has_frames"] = (int(stage1_summary["total_frames"]) > 0)
checks["all_eight_target_models_pass"] = (
    final_df[
        final_df["model"] != "YOLOv8_Stage1"
    ]["status"].eq("PASS").all()
)

target_rows = final_df[
    final_df["model"] != "YOLOv8_Stage1"
]
checks["all_target_frame_counts_match_stage1"] = bool(
    target_rows["total_frames"]
    .fillna(-1)
    .astype(int)
    .eq(int(stage1_summary["total_frames"]))
    .all()
)

# ViT preprocessing must match the independently reproduced image-level protocol.
vit_metric_path = (
    WORK_ROOT / "classifiers" / "ViT" / "metrics_summary.json"
)
vit_metrics = (
    json.loads(vit_metric_path.read_text(encoding="utf-8"))
    if vit_metric_path.exists()
    else {}
)
checks["vit_preprocessing_is_0p5_normalization"] = (
    vit_metrics.get("vit_preprocessing")
    == "Resize(224,224)->ToTensor()->Normalize(mean=[0.5]*3,std=[0.5]*3)"
)

# Classification TS must be inherited from one shared Stage-1 output.
classifier_rows = final_df[
    final_df["branch"] == "classification"
].copy()

checks["classification_ts_source_is_shared_stage1"] = bool(
    classifier_rows["TS_source"]
    .eq("shared_YOLOv8_stage1")
    .all()
)

shared_ts_pct = (
    100.0 * stage1_summary["TS_detected_pairs"]
    if stage1_summary["TS_detected_pairs"] is not None
    else None
)

if shared_ts_pct is None:
    checks["classification_ts_values_equal_shared_stage1"] = False
else:
    checks["classification_ts_values_equal_shared_stage1"] = bool(
        np.allclose(
            classifier_rows["TS_detected_pairs_pct"].astype(float),
            float(shared_ts_pct),
            rtol=0,
            atol=1e-9,
            equal_nan=False,
        )
    )

# Canonical prediction outputs should never retain the alias "del".
canonical_del_leaks = []

for model_name, _ in CLASSIFICATION_MODELS:
    p = WORK_ROOT / "classifiers" / model_name / "frame_level_outputs.csv"
    if not p.exists():
        canonical_del_leaks.append({
            "model": model_name,
            "reason": "missing_frame_output"
        })
        continue

    d = pd.read_csv(p, usecols=["pred_label"])
    n = int(
        d["pred_label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("del")
        .sum()
    )
    if n:
        canonical_del_leaks.append({
            "model": model_name,
            "canonical_del_rows": n
        })

for model_name, _ in DETECTION_MODELS:
    p = (
        WORK_ROOT
        / "detectors_two_stage"
        / model_name
        / "frame_level_outputs.csv"
    )
    if not p.exists():
        canonical_del_leaks.append({
            "model": model_name,
            "reason": "missing_frame_output"
        })
        continue

    d = pd.read_csv(p, usecols=["pred_label"])
    n = int(
        d["pred_label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("del")
        .sum()
    )
    if n:
        canonical_del_leaks.append({
            "model": model_name,
            "canonical_del_rows": n
        })

checks["no_del_alias_remains_in_canonical_predictions"] = (
    len(canonical_del_leaks) == 0
)

# Target-detector TS must use properly remapped boxes.
detector_rows = final_df[
    final_df["branch"] == "detection"
].copy()

checks["detector_box_mapping_is_documented_and_corrected"] = bool(
    detector_rows["target_box_mapping"]
    .fillna("")
    .str.contains(
        "scaled back to original ROI size",
        regex=False
    )
    .all()
)

# Check checkpoint class-name semantic compatibility.
if LABEL_AUDIT_CSV.exists():
    _audit = pd.read_csv(LABEL_AUDIT_CSV)
    checks["ultralytics_semantic_label_sets_match"] = bool(
        _audit["semantic_label_set_match"]
        .fillna(False)
        .all()
    )
else:
    checks["ultralytics_semantic_label_sets_match"] = False

# Timing is intentionally transparent rather than overstated.
checks["timing_scope_explicitly_labeled"] = bool(
    target_rows["timing_scope"]
    .fillna("")
    .str.contains(
        "component_sum_e2e_FPS_estimate",
        regex=False
    )
    .all()
)

checks["exact_wallclock_e2e_not_falsely_claimed"] = bool(
    (~target_rows[
        "exact_single_process_wallclock_e2e_measured"
    ].fillna(True).astype(bool)).all()
)

editor_ready = all(bool(v) for v in checks.values())

verification = {
    "evaluation_version": EVALUATION_VERSION,
    "editor_ready_for_submission": editor_ready,
    "checks": checks,
    "canonical_del_leaks": canonical_del_leaks,
    "scientific_interpretation": {
        "strict_FA": (
            "Primary frame-accuracy metric over all frames; a missing Stage-1 "
            "localization is counted as an incorrect strict prediction."
        ),
        "operational_nothing_policy": (
            "Supplemental metric only: when ground truth is 'nothing' and "
            "Stage 1 returns no hand, the operational system output is 'nothing'."
        ),
        "classification_TS": (
            "Computed from the shared YOLOv8 Stage-1 boxes and therefore is "
            "not an intrinsic classifier-specific stability metric."
        ),
        "detection_TS": (
            "Computed from target-detector boxes after scaling 224x224 ROI "
            "coordinates back to the original ROI and then to full-frame coordinates."
        ),
        "FPS": (
            "model_only_FPS is target inference only. "
            "component_sum_e2e_FPS_estimate is the sum of measured sequential "
            "components and is not an integrated single-process wall-clock E2E FPS."
        ),
        "detector_protocol_provenance": (
            "The post-publication re-evaluation implements the two-stage pipeline "
            "described in published Section 4.3. The notebook does not claim that "
            "historical execution provenance was independently recovered unless "
            "supported by separate archived logs."
        ),
    },
}

save_json(verification, FINAL_VERIFICATION_JSON)

display(
    pd.DataFrame(
        [{"check": k, "pass": bool(v)} for k, v in checks.items()]
    )
)

print("\nEDITOR READY:", editor_ready)
if not editor_ready:
    print(
        "⚠ One or more verification checks failed. "
        "Inspect IJIES_EDITOR_VIDEO_FINAL_VERIFICATION.json before submission."
    )

# ------------------------------------------------------------
# Editor-facing README
# ------------------------------------------------------------
README_PATH = WORK_ROOT / "README_EDITOR_EVIDENCE.txt"

README_PATH.write_text(
f"""IJIES POST-PUBLICATION VIDEO RE-EVALUATION EVIDENCE
======================================================

Evaluation version:
{EVALUATION_VERSION}

Primary files
-------------
- IJIES_EDITOR_VIDEO_FINAL_SUMMARY.csv
- IJIES_EDITOR_TABLE5_REEVALUATION_EVIDENCE.csv
- IJIES_EDITOR_LOCALIZATION_RECOGNITION_ERROR_ANALYSIS.csv
- IJIES_EDITOR_VIDEO_FINAL_VERIFICATION.json
- IJIES_VIDEO_MODEL_LABEL_AUDIT.csv
- IJIES_VIDEO_MODEL_MANIFEST.csv
- IJIES_VIDEO_INPUT_MANIFEST.csv
- stage1_yolov8/frame_level_stage1.csv
- per-model frame_level_outputs.csv and metrics_summary.json
- subprocess logs/
- efficientnet_compatibility_audit.json

Corrections enforced by this re-evaluation
------------------------------------------
1. ViT video preprocessing uses Resize(224,224), ToTensor, and
   Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]), matching the
   independently reproduced benchmark/cross-domain evaluation.
2. Semantic alias 'del' is canonicalized to 'delete'. Raw checkpoint
   class names are preserved in the label-audit files.
3. For target detectors, boxes predicted on a resized 224x224 ROI are
   scaled back to the original ROI dimensions before being mapped to
   full-frame coordinates for Temporal Stability.
4. Classification TS is explicitly sourced from the shared YOLOv8
   Stage-1 boxes and is not treated as classifier-specific.
5. Strict FA uses all video frames. Missing Stage-1 localization is a
   strict failure. The no-hand -> 'nothing' rule is reported only as a
   supplemental operational metric.
6. Timing scope is explicit. model_only_FPS measures the target model
   call. component_sum_e2e_FPS_estimate is a measured component sum;
   it is NOT labeled as exact integrated wall-clock E2E FPS.

YOLOv8 Stage-1 re-evaluation settings
-------------------------------------
checkpoint: {HAND_DETECTOR_PATH}
confidence threshold: {HAND_CONF}
imgsz: {HAND_IMGSZ}
frame stride: {FRAME_STRIDE}
checkpoint SHA256: {stage1_summary["weights_sha256"]}

Important provenance statement
------------------------------
The two-stage detector re-evaluation follows the pipeline described in
published Section 4.3. This notebook does not by itself establish the
historical execution settings unless separate archived logs are available.

Verification
------------
editor_ready_for_submission = {editor_ready}
See: IJIES_EDITOR_VIDEO_FINAL_VERIFICATION.json
""",
encoding="utf-8"
)

# ------------------------------------------------------------
# ZIP all generated evidence
# ------------------------------------------------------------
if FINAL_EVIDENCE_ZIP.exists():
    FINAL_EVIDENCE_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_EVIDENCE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in sorted(WORK_ROOT.rglob("*")):
        if p.is_file():
            z.write(
                p,
                Path("IJIES_EDITOR_VIDEO_EVIDENCE") / p.relative_to(WORK_ROOT)
            )

print("\nFinal summary:")
print(FINAL_SUMMARY_CSV)

print("\nVerification:")
print(FINAL_VERIFICATION_JSON)

print("\nEditor evidence ZIP:")
print(FINAL_EVIDENCE_ZIP)
